# CT Preprocessing Deep Dive

This notebook walks through every preprocessing step applied to CQ500 DICOM CT scans before they are fed into the model. Each step is explained and visualised so you can understand both **what** is happening and **why**.

### Steps covered
1. Data quality audit — find corrupted files, missing metadata
2. DICOM → Hounsfield Units (HU) — the raw-to-physical conversion
3. HU clipping — removing physically meaningless extremes
4. CT windowing — isolating tissue types of interest
5. Two input strategies compared — 3-window vs adjacent-slice stacking (paper method)
6. Pixel spacing and voxel resampling — correcting for scanner differences
7. Resizing to model input size
8. Intensity normalisation
9. Augmentation pipeline preview
10. Label distribution and class imbalance

## 0. Setup

In [3]:
import sys
sys.path.insert(0, '..')  # make src/ importable

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pydicom
import cv2

from src.data.preprocessing import (
    apply_window, clip_hu, hu_to_3channel,
    adjacent_slices_to_3channel, normalize_image,
    load_dicom_slice, CT_WINDOWS, HEMORRHAGE_TYPES
)

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

all_dcm = sorted(RAW_DIR.rglob('*.dcm'))
print(f'Total DICOM files found: {len(all_dcm)}')

KeyboardInterrupt: 

---
## 1. Data Quality Audit

Before any preprocessing, check the dataset for problems:
- Files that cannot be read (corrupted)
- Missing critical metadata (`RescaleSlope`, `PixelSpacing`, `SliceLocation`)
- Slices with unusual dimensions
- Slices with extreme or flat HU distributions (scan artifacts)

In [ ]:
# ── Audit all DICOM files ────────────────────────────────────────────────────
issues = []
rows   = []

for dcm_path in all_dcm:
    row = {'path': str(dcm_path)}
    try:
        dcm = pydicom.dcmread(str(dcm_path), stop_before_pixels=True)
        row['readable']        = True
        row['rows']            = getattr(dcm, 'Rows',            None)
        row['cols']            = getattr(dcm, 'Columns',         None)
        row['has_slope']       = hasattr(dcm, 'RescaleSlope')
        row['has_intercept']   = hasattr(dcm, 'RescaleIntercept')
        row['has_spacing']     = hasattr(dcm, 'PixelSpacing')
        row['has_location']    = hasattr(dcm, 'SliceLocation')
        row['pixel_spacing']   = list(getattr(dcm, 'PixelSpacing', [None, None]))
        row['slice_location']  = float(getattr(dcm, 'SliceLocation', 0))
    except Exception as e:
        row['readable'] = False
        issues.append(f'UNREADABLE: {dcm_path.name} — {e}')
    rows.append(row)

df_audit = pd.DataFrame(rows)

print('=== Quality Summary ===')
print(f'Total files          : {len(df_audit)}')
print(f'Unreadable           : {(~df_audit["readable"]).sum()}')
print(f'Missing RescaleSlope : {(~df_audit["has_slope"]).sum()}')
print(f'Missing PixelSpacing : {(~df_audit["has_spacing"]).sum()}')
print(f'Missing SliceLocation: {(~df_audit["has_location"]).sum()}')

dim_counts = df_audit[df_audit['readable']].groupby(['rows','cols']).size().reset_index(name='count')
print('\nImage dimensions found:')
print(dim_counts.to_string(index=False))

if issues:
    print('\nIssues detected:')
    for i in issues: print(' ', i)
else:
    print('\nNo issues detected — dataset looks clean.')

KeyboardInterrupt: 

In [ ]:
# ── Pixel spacing distribution across studies ────────────────────────────────
# Different scanners may produce different pixel spacings (mm per pixel).
# If spacing varies a lot, resampling to a uniform spacing is important.

spacings = []
for _, row in df_audit[df_audit['readable'] & df_audit['has_spacing']].iterrows():
    sp = row['pixel_spacing']
    if sp[0] is not None:
        spacings.append(float(sp[0]))

if spacings:
    spacings = np.array(spacings)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(spacings, bins=30, color='teal', edgecolor='none', alpha=0.8)
    ax.axvline(np.median(spacings), color='red', linestyle='--', label=f'Median = {np.median(spacings):.3f} mm')
    ax.set_xlabel('Pixel spacing (mm/pixel)', fontsize=12)
    ax.set_ylabel('# slices', fontsize=12)
    ax.set_title('Pixel spacing distribution across all slices', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f'Min: {spacings.min():.3f} mm  |  Max: {spacings.max():.3f} mm  |  Std: {spacings.std():.4f} mm')

---
## 2. DICOM → Hounsfield Units (HU)

DICOM scanners store raw pixel values — not physically meaningful numbers. Each file header contains two parameters to convert them:

$$\text{HU} = \text{pixel\_value} \times \text{RescaleSlope} + \text{RescaleIntercept}$$

**Reference HU values:**

| Tissue | Typical HU |
|---|---|
| Air | −1000 |
| Fat | −100 to −50 |
| Water / CSF | 0 |
| Grey matter | 37–45 |
| White matter | 20–30 |
| Acute blood | 50–80 |
| Bone | 400–1000 |

In [ ]:
# Load one example slice
sample_path = all_dcm[len(all_dcm) // 2]  # pick a middle slice (more likely to contain brain)
dcm = pydicom.dcmread(str(sample_path))

slope     = float(getattr(dcm, 'RescaleSlope', 1))
intercept = float(getattr(dcm, 'RescaleIntercept', 0))

raw_pixels = dcm.pixel_array.astype(np.float32)
hu_raw     = raw_pixels * slope + intercept

print(f'RescaleSlope    : {slope}')
print(f'RescaleIntercept: {intercept}')
print(f'Raw pixel range : {raw_pixels.min():.0f} → {raw_pixels.max():.0f}')
print(f'HU range (raw)  : {hu_raw.min():.0f} → {hu_raw.max():.0f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(raw_pixels, cmap='gray')
axes[0].set_title('Raw pixel values\n(scanner units, meaningless)', fontsize=11)
axes[0].axis('off')

axes[1].imshow(np.clip(hu_raw, -100, 200), cmap='gray')  # narrow clip just for display
axes[1].set_title('After HU conversion\n(physical units — brain tissue visible)', fontsize=11)
axes[1].axis('off')

plt.suptitle('Raw pixels vs Hounsfield Units', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. HU Clipping

CT scans contain many pixels outside the brain — including air outside the head and scanner bed material. These extreme HU values add noise and waste model capacity. We clip to a physiologically meaningful range: **-1000 to 3000 HU**.

The histogram below shows the before/after effect.

In [ ]:
hu_clipped = clip_hu(hu_raw, hu_min=-1000, hu_max=3000)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Reference lines
tissue_ref = {'Air': -1000, 'Fat': -80, 'CSF/Water': 0, 'Brain': 40,
               'Acute blood': 65, 'Bone edge': 400}

for ax, (hu_arr, title, xlim) in zip(axes, [
    (hu_raw,     'HU distribution — before clipping', (-2000, 5000)),
    (hu_clipped, 'HU distribution — after clipping',  (-1100, 3100)),
]):
    ax.hist(hu_arr.flatten(), bins=120, color='steelblue', edgecolor='none', alpha=0.8)
    for label, val in tissue_ref.items():
        ax.axvline(val, color='red', linestyle='--', linewidth=0.8)
        ax.text(val + 20, ax.get_ylim()[1] * 0.85, label, fontsize=7, color='darkred', rotation=90, va='top')
    ax.set_xlabel('HU', fontsize=11)
    ax.set_ylabel('Pixel count', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(xlim)

plt.tight_layout()
plt.show()

---
## 4. CT Windowing — Isolating Tissue Types

Even after converting to HU, the full range (−1000 to 3000) is too wide to display or train on directly. **Windowing** selects a sub-range to focus on:

$$\text{pixel}_{\text{display}} = \frac{\text{HU} - (\text{center} - \text{width}/2)}{\text{width}} \quad \text{clipped to } [0, 1]$$

The three standard windows used in this project:

| Window | Center | Width | Purpose |
|---|---|---|---|
| Brain | 40 | 80 | Differentiates grey/white matter and acute blood |
| Subdural | 75 | 215 | Optimal for subdural and epidural haematomas |
| Bone | 600 | 2800 | Shows skull fractures and calcification |

In [ ]:
# ── Standard 3-window comparison ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, (center, width)) in zip(axes, CT_WINDOWS.items()):
    windowed = apply_window(hu_clipped, center, width)
    ax.imshow(windowed, cmap='gray')
    ax.set_title(f'{name.capitalize()} window\nC={center} HU, W={width} HU', fontsize=12)
    ax.axis('off')

plt.suptitle('CT windowing comparison — same slice', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Effect of varying window center and width ────────────────────────────────
# This shows why choosing the right window matters.

centers = [0, 40, 75, 150]
widths  = [80, 80, 215, 300]
labels  = ['Water-level\n(C=0, W=80)', 'Brain\n(C=40, W=80)',
           'Subdural\n(C=75, W=215)', 'Soft tissue\n(C=150, W=300)']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, c, w, lbl in zip(axes, centers, widths, labels):
    ax.imshow(apply_window(hu_clipped, c, w), cmap='gray')
    ax.set_title(lbl, fontsize=11)
    ax.axis('off')

plt.suptitle('Effect of different window center / width choices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Two Input Strategies: 3-Window vs Adjacent-Slice Stacking

There are two ways to create a 3-channel (RGB-like) image from CT data:

**Strategy A — 3-window (original EfficientNet/ResNet approach)**  
Apply three different windows to the *same* slice → channels = brain, subdural, bone

**Strategy B — Adjacent-slice stacking (paper method, used in this project)**  
Apply the *same* brain window to three consecutive slices → channels = slice(s−1), slice(s), slice(s+1)  
This gives the model **implicit 3D context** — it can see how anatomy changes between slices.

In [ ]:
# Load three consecutive slices from the same study
def load_hu(path):
    d = pydicom.dcmread(str(path))
    px = d.pixel_array.astype(np.float32)
    slope = float(getattr(d, 'RescaleSlope', 1))
    intercept = float(getattr(d, 'RescaleIntercept', 0))
    return clip_hu(px * slope + intercept)

# Pick a consecutive triplet from the middle of the study
mid = len(all_dcm) // 2
hu_prev = load_hu(all_dcm[mid - 1]) if mid > 0 else None
hu_curr = load_hu(all_dcm[mid])
hu_next = load_hu(all_dcm[mid + 1]) if mid < len(all_dcm) - 1 else None

strategy_a = hu_to_3channel(hu_curr)                                  # 3-window
strategy_b = adjacent_slices_to_3channel(hu_prev, hu_curr, hu_next)   # adjacent-slice

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
ch_names_a = ['Brain window', 'Subdural window', 'Bone window']
ch_names_b = ['Prev slice (s−1)', 'Current slice (s)', 'Next slice (s+1)']

for i in range(3):
    axes[0, i].imshow(strategy_a[:, :, i], cmap='gray')
    axes[0, i].set_title(f'Ch {i+1}: {ch_names_a[i]}', fontsize=10)
    axes[0, i].axis('off')

    axes[1, i].imshow(strategy_b[:, :, i], cmap='gray')
    axes[1, i].set_title(f'Ch {i+1}: {ch_names_b[i]}', fontsize=10)
    axes[1, i].axis('off')

axes[0, 3].imshow(strategy_a)
axes[0, 3].set_title('RGB composite\n(Strategy A)', fontsize=10)
axes[0, 3].axis('off')

axes[1, 3].imshow(strategy_b)
axes[1, 3].set_title('RGB composite\n(Strategy B — paper method)', fontsize=10)
axes[1, 3].axis('off')

axes[0, 0].set_ylabel('Strategy A\n3-window', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Strategy B\nAdjacent-slice (paper)', fontsize=12, fontweight='bold')

plt.suptitle('Input strategy comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Strategy A shape: {strategy_a.shape}  |  value range: [{strategy_a.min():.3f}, {strategy_a.max():.3f}]')
print(f'Strategy B shape: {strategy_b.shape}  |  value range: [{strategy_b.min():.3f}, {strategy_b.max():.3f}]')

---
## 6. Pixel Spacing and Voxel Resampling

Different CT scanners — or different scan protocols — produce images with different pixel spacings (mm per pixel). A pixel in one study might represent 0.4 mm while in another it represents 0.7 mm. If we just resize all images to 384×384, structures will appear at different physical scales.

**Resampling** rescales every image to a uniform pixel spacing (e.g. 0.5 mm/pixel) so that anatomical structures appear at the same size regardless of scanner settings. This is especially important when comparing lesion sizes.

In [ ]:
def resample_to_spacing(hu_array: np.ndarray,
                        current_spacing: float,
                        target_spacing: float = 0.5) -> np.ndarray:
    """
    Rescale a 2D HU array from current_spacing to target_spacing (mm/pixel).
    Uses bicubic interpolation to preserve fine structure.
    """
    scale = current_spacing / target_spacing
    new_h = int(round(hu_array.shape[0] * scale))
    new_w = int(round(hu_array.shape[1] * scale))
    return cv2.resize(hu_array, (new_w, new_h), interpolation=cv2.INTER_CUBIC)

# Get actual pixel spacing from the sample DICOM
current_spacing = float(getattr(dcm, 'PixelSpacing', [0.488])[0])
print(f'Original pixel spacing : {current_spacing:.3f} mm/pixel')
print(f'Original image size    : {hu_clipped.shape}')

resampled_05 = resample_to_spacing(hu_clipped, current_spacing, target_spacing=0.5)
resampled_10 = resample_to_spacing(hu_clipped, current_spacing, target_spacing=1.0)

print(f'Resampled to 0.5 mm/px : {resampled_05.shape}')
print(f'Resampled to 1.0 mm/px : {resampled_10.shape}')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (arr, title) in zip(axes, [
    (hu_clipped,    f'Original\n({current_spacing:.3f} mm/px, {hu_clipped.shape[0]}×{hu_clipped.shape[1]})'),
    (resampled_05,  f'Resampled to 0.5 mm/px\n({resampled_05.shape[0]}×{resampled_05.shape[1]})'),
    (resampled_10,  f'Resampled to 1.0 mm/px\n({resampled_10.shape[0]}×{resampled_10.shape[1]})'),
]):
    ax.imshow(apply_window(arr, 40, 80), cmap='gray')
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.suptitle('Effect of resampling to uniform pixel spacing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Resizing to Model Input Size

The DenseNet121 backbone expects a fixed input size of **384×384 pixels** (set in `params.yaml`). Two common strategies:

- **Direct resize** — stretch/squish to 384×384. Fast, but distorts aspect ratio if the CT is not square.
- **Pad then resize** — add black borders to make the image square first, then resize. Preserves aspect ratio.

Brain CT scans are already square (rows = cols) in CQ500, so direct resize is fine here.

In [ ]:
TARGET_SIZE = 384

brain_win = apply_window(hu_clipped, 40, 80)  # [0, 1] float32

# Strategy 1: direct resize
resized_direct = cv2.resize(brain_win, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_LINEAR)

# Strategy 2: square-pad then resize
h, w = brain_win.shape
pad_size = max(h, w)
padded = np.zeros((pad_size, pad_size), dtype=np.float32)
padded[(pad_size - h) // 2 : (pad_size - h) // 2 + h,
       (pad_size - w) // 2 : (pad_size - w) // 2 + w] = brain_win
resized_padded = cv2.resize(padded, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_LINEAR)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (img, title) in zip(axes, [
    (brain_win,      f'Original\n{h}×{w}'),
    (resized_direct, f'Direct resize\n{TARGET_SIZE}×{TARGET_SIZE}'),
    (resized_padded, f'Pad → resize\n{TARGET_SIZE}×{TARGET_SIZE}'),
]):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.suptitle(f'Resize strategies → {TARGET_SIZE}×{TARGET_SIZE} model input', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Intensity Normalisation

After windowing, pixel values are in [0, 1]. Neural networks train better when inputs are centred around 0 with unit variance. We apply **ImageNet-style normalisation**:

$$x_{\text{norm}} = \frac{x - \mu}{\sigma}$$

Where μ and σ are the ImageNet channel means and standard deviations. The paper uses a single-channel version: μ=0.456, σ=0.224.

In [ ]:
# Create a 3-channel image (adjacent-slice, paper method)
img_3ch = adjacent_slices_to_3channel(hu_prev, hu_curr, hu_next)  # shape (H, W, 3), values in [0,1]

# Resize to model input
img_resized = cv2.resize(img_3ch, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_LINEAR)

# Paper normalisation (single mean/std applied to all 3 channels)
paper_mean = np.array([0.456, 0.456, 0.456], dtype=np.float32)
paper_std  = np.array([0.224, 0.224, 0.224], dtype=np.float32)
img_paper_norm = (img_resized - paper_mean) / paper_std

# ImageNet normalisation
imagenet_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
imagenet_std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
img_imagenet_norm = (img_resized - imagenet_mean) / imagenet_std

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (img, title, stats) in zip(axes, [
    (img_resized,       'After resize\n[0,1]',   img_resized),
    (img_paper_norm,    'Paper norm\nμ=0.456, σ=0.224', img_paper_norm),
    (img_imagenet_norm, 'ImageNet norm\n(per-channel)', img_imagenet_norm),
]):
    # Clip for display only (normalised values go negative)
    display = np.clip((stats - stats.min()) / (stats.max() - stats.min() + 1e-8), 0, 1)
    ax.imshow(display)
    ax.set_title(f'{title}\nmean={stats.mean():.3f}  std={stats.std():.3f}', fontsize=10)
    ax.axis('off')

plt.suptitle('Intensity normalisation comparison (paper method uses μ=0.456)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Augmentation Pipeline Preview

Augmentation adds variety to training data to prevent overfitting. The paper (`aug_image`) applies:
1. **Horizontal flip** (p=0.5) — brain hemorrhages can appear on either side
2. **Shift, scale, rotate** — small geometric perturbations
3. **Random erasing** (p=0.5) — randomly masks a rectangular region, forcing the model to focus on context rather than a single spot
4. **Random crop** (60–99% of image) — simulates slightly different FOV

Augmentation is applied **only during training**, never during validation or test.

In [ ]:
import numpy as np
from src.data.augmentation import build_paper_train_transforms, build_paper_val_transforms

train_transform = build_paper_train_transforms(image_size=TARGET_SIZE)
val_transform   = build_paper_val_transforms(image_size=TARGET_SIZE)

# The transform expects a uint8 or float32 (H, W, 3) image
base_img = img_resized.copy()  # float32, [0,1], (384,384,3)

# Generate multiple augmented versions
n_samples = 8
augmented = [train_transform(image=base_img)['image'].numpy() for _ in range(n_samples)]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (ax, aug) in enumerate(zip(axes, augmented)):
    # aug is a torch tensor (C, H, W) — transpose for display
    img_display = np.transpose(aug, (1, 2, 0))
    img_display = np.clip((img_display - img_display.min()) /
                          (img_display.max() - img_display.min() + 1e-8), 0, 1)
    ax.imshow(img_display)
    ax.set_title(f'Augmented sample {i+1}', fontsize=10)
    ax.axis('off')

plt.suptitle('Training augmentation — 8 random variations of the same slice', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Train vs Val transform comparison ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

val_out = val_transform(image=base_img)['image'].numpy()
val_out_disp = np.transpose(val_out, (1, 2, 0))
val_out_disp = np.clip((val_out_disp - val_out_disp.min()) /
                       (val_out_disp.max() - val_out_disp.min() + 1e-8), 0, 1)

aug_out = train_transform(image=base_img)['image'].numpy()
aug_out_disp = np.transpose(aug_out, (1, 2, 0))
aug_out_disp = np.clip((aug_out_disp - aug_out_disp.min()) /
                       (aug_out_disp.max() - aug_out_disp.min() + 1e-8), 0, 1)

axes[0].imshow(base_img)
axes[0].set_title('Original (resized)', fontsize=11)
axes[0].axis('off')

axes[1].imshow(val_out_disp)
axes[1].set_title('Val transform\n(resize + centre crop + normalise)', fontsize=11)
axes[1].axis('off')

axes[2].imshow(aug_out_disp)
axes[2].set_title('Train transform\n(+ flip + rotate + erase + crop)', fontsize=11)
axes[2].axis('off')

plt.suptitle('Train vs Validation transform (val is deterministic — same every time)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Label Distribution and Class Imbalance

Hemorrhage is relatively rare. Most CT scans show no hemorrhage at all (`no_hemorrhage = 1`). Among positive cases, subtypes vary in frequency. Understanding this imbalance is critical for interpreting model performance — a model that predicts "no hemorrhage" for every scan would achieve high accuracy but be clinically useless.

In [ ]:
# Load processed CSVs (created by make prepare)
split_stats = {}
for split in ['train', 'val', 'test']:
    csv_path = PROCESSED_DIR / f'{split}.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        split_stats[split] = df
        print(f'{split:5s}: {len(df):>6,} slices')
    else:
        print(f'{split:5s}: CSV not found — run `make prepare` first')

if split_stats:

In [ ]:
if split_stats:
    df_all = pd.concat(split_stats.values(), ignore_index=True)

    label_cols = [c for c in HEMORRHAGE_TYPES if c in df_all.columns]

    # ── Positive rate per label ──────────────────────────────────────────────
    pos_rates = df_all[label_cols].mean() * 100  # percentage

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    colors = ['#2ecc71' if l == 'no_hemorrhage' else '#e74c3c' for l in label_cols]
    bars = axes[0].bar(label_cols, pos_rates, color=colors, edgecolor='white')
    axes[0].set_ylabel('% of slices positive', fontsize=12)
    axes[0].set_title('Positive rate per class (all splits combined)', fontsize=12, fontweight='bold')
    axes[0].set_xticklabels(label_cols, rotation=20, ha='right')
    for bar, val in zip(bars, pos_rates):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

    # ── Per-split comparison ─────────────────────────────────────────────────
    x = np.arange(len(label_cols))
    w = 0.25
    split_colors = {'train': '#3498db', 'val': '#e67e22', 'test': '#9b59b6'}
    for i, (split, df) in enumerate(split_stats.items()):
        rates = [df[col].mean() * 100 if col in df.columns else 0 for col in label_cols]
        axes[1].bar(x + i * w, rates, w, label=split, color=split_colors[split], alpha=0.85)

    axes[1].set_xticks(x + w)
    axes[1].set_xticklabels(label_cols, rotation=20, ha='right')
    axes[1].set_ylabel('% positive', fontsize=12)
    axes[1].set_title('Label distribution per split', fontsize=12, fontweight='bold')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    # ── Raw counts table ─────────────────────────────────────────────────────
    counts = {split: df[label_cols].sum().astype(int).rename(split)
              for split, df in split_stats.items() if label_cols[0] in df.columns}
    print('\nPositive slice counts per label:')
    print(pd.DataFrame(counts).to_string())
else:
    print('Run `make prepare` to generate the CSV files, then re-run this cell.')

---
## 11. Data Balancing

Brain CT scans are heavily imbalanced in two ways:

1. **Study-level imbalance** — most studies show no hemorrhage at all
2. **Slice-level imbalance** — even in a positive study, only a small fraction of slices actually contain hemorrhage (the top/bottom of the head contains skull/air, not brain)

There are several strategies to address this. This section visualises and compares them.

| Strategy | How it works | Used in this project? |
|---|---|---|
| **Do nothing** | Train on all slices equally | No — model learns to always predict negative |
| **Class weights in loss** | Multiply loss more for minority classes | Partially — `pos_weight` in BCEWithLogitsLoss |
| **WeightedRandomSampler** | Over-sample positive slices in DataLoader | Available — shown below |
| **Study-based sampling** | Sample one slice per study per epoch | ✅ Paper method (`ICHStudyDataset`) |
| **Centre-weighted slice sampling** | Within a study, favour middle slices over skull-cap | ✅ Paper method (`_build_weighted_index_list`) |

In [ ]:
# ── 11a. Visualise the imbalance problem ────────────────────────────────────
# Show side-by-side: what a naive sampler sees vs. a balanced sampler

if not split_stats:
    print("Run `make prepare` first to generate CSV files.")
else:
    df_train = split_stats['train']
    label_cols = [c for c in HEMORRHAGE_TYPES if c in df_train.columns]

    total = len(df_train)
    n_negative = (df_train['no_hemorrhage'] == 1).sum()
    n_positive = total - n_negative

    print(f"Training set: {total:,} total slices")
    print(f"  No hemorrhage : {n_negative:,} ({100*n_negative/total:.1f}%)")
    print(f"  Any hemorrhage: {n_positive:,} ({100*n_positive/total:.1f}%)")
    print()

    # Per-subtype counts (excluding no_hemorrhage)
    ich_cols = [c for c in label_cols if c != 'no_hemorrhage']
    for col in ich_cols:
        n = df_train[col].sum()
        print(f"  {col:25s}: {n:>6,}  ({100*n/total:.2f}%)")

    # Pie chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].pie([n_negative, n_positive],
                labels=['No hemorrhage', 'Any hemorrhage'],
                colors=['#2ecc71', '#e74c3c'],
                autopct='%1.1f%%', startangle=90,
                wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    axes[0].set_title('Study-level balance\n(train split)', fontsize=12, fontweight='bold')

    counts = [df_train[c].sum() for c in ich_cols]
    colors_ich = ['#e74c3c', '#e67e22', '#9b59b6', '#3498db', '#1abc9c']
    axes[1].bar(ich_cols, counts, color=colors_ich, edgecolor='white')
    axes[1].set_ylabel('Positive slice count', fontsize=11)
    axes[1].set_xticklabels(ich_cols, rotation=20, ha='right')
    axes[1].set_title('Hemorrhage subtype distribution\n(train split)', fontsize=12, fontweight='bold')
    for i, v in enumerate(counts):
        axes[1].text(i, v + max(counts)*0.01, str(v), ha='center', fontsize=9)

    plt.tight_layout()
    plt.show()

### Strategy 1: WeightedRandomSampler (PyTorch)

PyTorch's `WeightedRandomSampler` assigns a sampling probability to each slice. Positive slices get a higher weight so they are drawn more often during training. The DataLoader then draws a balanced mini-batch even though the dataset itself is imbalanced.

In [ ]:
# ── 11b. WeightedRandomSampler ───────────────────────────────────────────────
import torch
from torch.utils.data import WeightedRandomSampler

if not split_stats:
    print("Run `make prepare` first.")
else:
    df_train = split_stats['train']
    label_cols = [c for c in HEMORRHAGE_TYPES if c in df_train.columns]
    ich_cols = [c for c in label_cols if c != 'no_hemorrhage']

    # A slice is "positive" if any hemorrhage subtype == 1
    is_positive = (df_train[ich_cols].sum(axis=1) > 0).astype(int)

    n_pos = is_positive.sum()
    n_neg = len(is_positive) - n_pos

    # Weight: inverse class frequency  →  positive slices sampled more often
    weight_pos = 1.0 / n_pos if n_pos > 0 else 0
    weight_neg = 1.0 / n_neg if n_neg > 0 else 0
    sample_weights = is_positive.map({1: weight_pos, 0: weight_neg}).values

    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.float),
        num_samples=len(sample_weights),
        replacement=True,
    )

    print(f"Positive slices : {n_pos:,}  (weight = {weight_pos:.6f})")
    print(f"Negative slices : {n_neg:,}  (weight = {weight_neg:.6f})")
    print(f"Weight ratio pos/neg: {weight_pos/weight_neg:.1f}×  "
          f"(positive samples drawn ~{weight_pos/weight_neg:.0f}× more often)")

    # ── Simulate what one epoch looks like with vs without sampler ───────────
    np.random.seed(42)
    epoch_size = min(10_000, len(is_positive))

    # Naive (no sampler) — sample with replacement, uniform probability
    naive_idx = np.random.choice(len(is_positive), size=epoch_size, replace=True)
    naive_pos_rate = is_positive.values[naive_idx].mean() * 100

    # Weighted sampler — sample with replacement, proportional to weight
    weighted_idx = np.random.choice(
        len(is_positive), size=epoch_size, replace=True,
        p=sample_weights / sample_weights.sum()
    )
    weighted_pos_rate = is_positive.values[weighted_idx].mean() * 100

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, (label, rate) in zip(axes, [
        ('Naive (no sampler)',      naive_pos_rate),
        ('WeightedRandomSampler',   weighted_pos_rate),
    ]):
        ax.bar(['Negative', 'Positive'], [100 - rate, rate],
               color=['#2ecc71', '#e74c3c'], edgecolor='white')
        ax.set_ylabel('% of drawn samples', fontsize=11)
        ax.set_title(f'{label}\n{rate:.1f}% positive in each epoch batch', fontsize=11, fontweight='bold')
        ax.set_ylim(0, 105)
        for i, v in enumerate([100 - rate, rate]):
            ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=10)

    plt.suptitle('Naive sampling vs WeightedRandomSampler — simulated epoch composition',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

### Strategy 2: Class Weights in Loss Function (`pos_weight`)

Instead of changing *which* samples are drawn, we can change *how much each prediction error costs*. `BCEWithLogitsLoss` accepts a `pos_weight` tensor — a multiplier applied to the loss on positive examples only.

$$\text{loss} = -\left[ w_{\text{pos}} \cdot y \cdot \log\hat{y} + (1-y) \cdot \log(1-\hat{y}) \right]$$

Setting `pos_weight = n_neg / n_pos` for each class makes the loss treat one positive example as equivalent to `n_neg/n_pos` negative examples.

In [ ]:
# ── 11c. Compute pos_weight for each class ───────────────────────────────────
if not split_stats:
    print("Run `make prepare` first.")
else:
    df_train = split_stats['train']
    label_cols = [c for c in HEMORRHAGE_TYPES if c in df_train.columns]
    total = len(df_train)

    print(f"{'Class':<28} {'n_pos':>7} {'n_neg':>8} {'pos_weight':>12}  (= n_neg / n_pos)")
    print("-" * 65)

    pos_weights = []
    for col in label_cols:
        n_pos = int(df_train[col].sum())
        n_neg = total - n_pos
        pw = n_neg / n_pos if n_pos > 0 else 0.0
        pos_weights.append(pw)
        print(f"  {col:<26} {n_pos:>7,} {n_neg:>8,} {pw:>12.2f}")

    print()
    print("Paste into params.yaml under loss.pos_weight:")
    print(f"  pos_weight: {[round(w, 2) for w in pos_weights]}")

    # Visualise
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#2ecc71' if w < 5 else '#e67e22' if w < 50 else '#e74c3c' for w in pos_weights]
    bars = ax.bar(label_cols, pos_weights, color=colors, edgecolor='white')
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, label='pos_weight = 1 (balanced)')
    ax.set_ylabel('pos_weight  (n_neg / n_pos)', fontsize=11)
    ax.set_title('Recommended pos_weight per class\n(higher = more emphasis on that class in loss)',
                 fontsize=12, fontweight='bold')
    ax.set_xticklabels(label_cols, rotation=20, ha='right')
    ax.legend()
    for bar, val in zip(bars, pos_weights):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(pos_weights)*0.01,
                f'{val:.1f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.show()

### Strategy 3: Study-based Sampling + Centre-weighted Slice Selection (Paper Method)

This is what the project actually uses. Instead of operating at the slice level, it operates at the **study level**:

1. **Study-based sampling** — each epoch draws one slice per CT study. This naturally prevents a 400-slice scan from dominating over a 100-slice scan (which would happen with flat slice-level sampling).

2. **Centre-weighted slice selection** — within a study, the slice to train on is chosen randomly but with higher probability given to middle slices. The top and bottom of the head contain mostly skull-cap and skull-base (no brain tissue, therefore never hemorrhage). Training on those wastes capacity.

The weight follows this rule (from `_build_weighted_index_list`):
```
weight(i) = max(1, i // 4)          if i <= length/2   (ramp up toward centre)
weight(i) = max(1, (length-i) // 4) if i >  length/2   (ramp down away from centre)
```

In [ ]:
# ── 11d. Visualise centre-weighted slice sampling ────────────────────────────
from src.data.dataset import _build_weighted_index_list

# Simulate a 200-slice study
study_length = 200
weighted_list = _build_weighted_index_list(study_length)

# Count how often each slice index appears in the weighted list
sampling_freq = np.zeros(study_length)
for idx in weighted_list:
    sampling_freq[idx] += 1
sampling_prob = sampling_freq / sampling_freq.sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(study_length), sampling_prob, color='steelblue', width=1.0, edgecolor='none')
axes[0].set_xlabel('Slice index (0 = top of head, 199 = bottom)', fontsize=11)
axes[0].set_ylabel('Sampling probability', fontsize=11)
axes[0].set_title('Centre-weighted slice sampling\n(paper method, 200-slice study)', fontsize=12, fontweight='bold')
axes[0].axvspan(0, 20, alpha=0.15, color='red', label='Skull-cap (rarely informative)')
axes[0].axvspan(180, 200, alpha=0.15, color='red', label='Skull-base (rarely informative)')
axes[0].axvspan(60, 140, alpha=0.1, color='green', label='Brain centre (most informative)')
axes[0].legend(fontsize=9)

# Compare: uniform vs centre-weighted for 3 different study lengths
for length, color in [(50, '#3498db'), (150, '#e67e22'), (300, '#9b59b6')]:
    wl = _build_weighted_index_list(length)
    freq = np.zeros(length)
    for idx in wl: freq[idx] += 1
    prob = freq / freq.sum()
    norm_pos = np.linspace(0, 1, length)
    axes[1].plot(norm_pos, prob, color=color, alpha=0.8, label=f'{length} slices')

axes[1].axhline(1/100, color='gray', linestyle='--', linewidth=1, label='Uniform (100-slice study)')
axes[1].set_xlabel('Normalised slice position (0=top, 1=bottom)', fontsize=11)
axes[1].set_ylabel('Sampling probability', fontsize=11)
axes[1].set_title('Centre-weighting for different study lengths', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Weighted list length (200-slice study): {len(weighted_list)}")
print(f"  Min sampling prob: {sampling_prob.min():.5f}  (edge slices)")
print(f"  Max sampling prob: {sampling_prob.max():.5f}  (centre slices)")
print(f"  Ratio centre/edge: {sampling_prob.max()/max(sampling_prob.min(),1e-9):.1f}×")

### Balancing Strategy Comparison Summary

| Strategy | Pros | Cons | Used here? |
|---|---|---|---|
| **Naive (no balancing)** | Simple | Model learns to predict "no hemorrhage" always | ❌ |
| **WeightedRandomSampler** | Easy to apply; works at slice level | Repeats minority slices many times → potential overfitting | Optional |
| **pos_weight in BCELoss** | No data duplication; differentiable | Does not change which slices are seen, only penalisation | ✅ Configurable via `params.yaml` |
| **Study-based sampling** | Each patient seen equally; no study dominates | Requires grouping by study | ✅ Paper method (default) |
| **Centre-weighted slice selection** | Filters out uninformative skull slices | Slight info loss from edge slices | ✅ Paper method (default) |

**Recommendation:** Use all three together — study-based + centre-weighted sampling (already active), plus tuned `pos_weight` values in `params.yaml` (computed in the cell above).

---
## Summary

The full preprocessing pipeline applied to every slice before it reaches the model:

```
DICOM file (.dcm)
    │
    ▼ load_dicom_slice()
Raw pixel array  ──×slope + intercept──►  Hounsfield Units (HU)
    │
    ▼ clip_hu(−1000, 3000)
Clipped HU  ──removes scanner noise outside physiological range
    │
    ▼ adjacent_slices_to_3channel()          ← paper method
3-channel float32 image  [H, W, 3]  values in [0, 1]
  Ch0 = brain window applied to slice(s−1)
  Ch1 = brain window applied to slice(s)      ← primary slice
  Ch2 = brain window applied to slice(s+1)
    │
    ▼ cv2.resize() to 384×384
    │
    ▼ PaperTrainTransform  (train only)
        HFlip(p=0.5) → ShiftScaleRotate → RandomErase(p=0.5) → RandomCrop
    │  (or PaperValTransform for val/test: centre crop only)
    │
    ▼ Normalize(mean=0.456, std=0.224) → ToTensorV2
torch.Tensor  [3, 384, 384]  ← fed to DenseNet121
```